<a href="https://colab.research.google.com/github/melaniedaniel7/CFPB-NLP-Topic-Analysis/blob/main/CFPB_NLP_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Imported the dataset
from google.colab import files

uploaded = files.upload()

Saving complaints-2026-06-17_10_50.csv to complaints-2026-06-17_10_50.csv


In [5]:
# Checked the dataset uploaded correctly
import pandas as pd

df = pd.read_csv("complaints-2026-06-17_10_50.csv")
df.head()
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='object')

In [6]:
# Checked if any complaint narratives were missing
df.shape
df['Consumer complaint narrative'].isnull().sum()

# Dataset only contains complaint narratives now
complaints = df[['Consumer complaint narrative']].copy()
complaints = complaints.dropna()
complaints.shape

(14965, 1)

In [7]:
# Viewed an example complaint
# Realised that I have to clean the redacted text, represented by "XXX's", when doing text cleaning
complaints.head()
print(complaints['Consumer complaint narrative'].iloc[0])

I was told by XXXX XXXX that XXXX XXXX XXXX  gave them information about the status of my account and whether it was paid off or not and how much I owed. I didnt authorize that information to be given to any one.


In [10]:
# Import libraries and download NLTK resources for text cleaning
import nltk
# These libraries were not mentioned in phase 1 and were an additional add-on for efficient text preprocessing
import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Punkt tokenizer for word tokenization
nltk.download('punkt')
# Stopwords list in multiple languages
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
nltk.download('wordnet') # Added to resolve LookupError for WordNetLemmatizer

# Create a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))

# Lemmatizer converts words to their base/root form
lemmatizer = WordNetLemmatizer()

# Function to clean the text by converting text to lowercase and removing punctuation
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text)
        return text
    else:
        return text

# Function to tokenize the text into individual words
def tokenize_text(text):
    if isinstance(text, str):
        tokens = word_tokenize(text)
        tokens = [
            word for word in tokens
            if not set(word.lower()) == {'x'}
        ]
        return tokens
    else:
        return []

# Function to remove stopwords from the tokenized text
def remove_stopwords(tokens):
    if isinstance(tokens, list):
        return [word for word in tokens if word not in stop_words]
    else:
        return tokens

# Function to apply lemmatization to the tokens
def lemmatize_tokens(tokens):
    if isinstance(tokens, list):
        return [lemmatizer.lemmatize(token) for token in tokens]
    else:
        return tokens

# Test text cleaning techniques on a single consumer complaint before applying text cleaning to the entire dataset
# Used the same complaint example as earlier
sample = df['Consumer complaint narrative'].iloc[0]
print("Original:")
print(sample)
# Clean example complaint
cleaned = clean_text(sample)
print("\nCleaned:")
print(cleaned)
# Tokenize example complaint
tokens = tokenize_text(cleaned)
print("\nTokens:")
print(tokens)
# Remove stopwords from example complaint
filtered = remove_stopwords(tokens)
print("\nNO Stpwords:")
print(filtered)
# Lemmatize example complaint
final = lemmatize_tokens(filtered)
print("\nLemmatized:")
print(final)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


Original:
I was told by XXXX XXXX that XXXX XXXX XXXX  gave them information about the status of my account and whether it was paid off or not and how much I owed. I didnt authorize that information to be given to any one.

Cleaned:
i was told by xxxx xxxx that xxxx xxxx xxxx gave them information about the status of my account and whether it was paid off or not and how much i owed i didnt authorize that information to be given to any one

Tokens:
['i', 'was', 'told', 'by', 'that', 'gave', 'them', 'information', 'about', 'the', 'status', 'of', 'my', 'account', 'and', 'whether', 'it', 'was', 'paid', 'off', 'or', 'not', 'and', 'how', 'much', 'i', 'owed', 'i', 'didnt', 'authorize', 'that', 'information', 'to', 'be', 'given', 'to', 'any', 'one']

NO Stpwords:
['told', 'gave', 'information', 'status', 'account', 'whether', 'paid', 'much', 'owed', 'didnt', 'authorize', 'information', 'given', 'one']

Lemmatized:
['told', 'gave', 'information', 'status', 'account', 'whether', 'paid', 'much', 